In [ ]:
!nvidia-smi

In [ ]:
!pip install -q \
    transformers \
    scikit-learn \
    pandas \
    numpy \
    matplotlib \
    tqdm \
    pillow \
    adversarial-robustness-toolbox \
    datasets \
    accelerate

In [ ]:
import torch
import torchvision
import transformers
import sklearn
import pandas as pd
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print("MedSTRIDE-AI Experimental Environment")
print("=" * 50)
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)
print("=" * 50)

In [ ]:
!pip install -q wfdb pandas numpy scipy scikit-learn matplotlib tqdm

In [ ]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("WFDB:", wfdb.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import wfdb
import os

DATA_DIR = "/content/mitdb"

os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading MIT-BIH Arrhythmia Database...")

wfdb.dl_database(
    "mitdb",
    dl_dir=DATA_DIR
)

print("\nDownload complete!")
print("Files:", len(os.listdir(DATA_DIR)))
print(os.listdir(DATA_DIR)[:20])

In [ ]:
record = wfdb.rdrecord(
    "/content/mitdb/100"
)

annotation = wfdb.rdann(
    "/content/mitdb/100",
    "atr"
)

print("Signal shape:", record.p_signal.shape)
print("Sampling frequency:", record.fs)
print("Channels:", record.sig_name)
print("Number of annotations:", len(annotation.sample))
print("First 20 beat annotations:")
print(annotation.symbol[:20])

In [ ]:
signal = record.p_signal[:, 0]
fs = record.fs

seconds = 10
samples = int(seconds * fs)

time = np.arange(samples) / fs

plt.figure(figsize=(15, 4))
plt.plot(time, signal[:samples])
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.title("MIT-BIH Record 100 — ECG Signal")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import os
import wfdb
import numpy as np
import pandas as pd
from collections import Counter

DATA_DIR = "/content/mitdb"

# MIT-BIH records
RECORDS = [
    '100','101','102','103','104','105','106','107','108','109',
    '111','112','113','114','115','116','117','118','119',
    '121','122','123','124',
    '200','201','202','203','205','207','208','209','210',
    '212','213','214','215','217','219','220','221','222',
    '223','228','230','231','232','233','234'
]

# AAMI-inspired beat grouping
NORMAL = {'N', 'L', 'R', 'e', 'j'}

ABNORMAL = {
    'A', 'a', 'J', 'S',        # supraventricular ectopic
    'V', 'E',                  # ventricular ectopic
    'F'                        # fusion
}

print("Number of records:", len(RECORDS))
print("Normal symbols:", NORMAL)
print("Abnormal symbols:", ABNORMAL)

In [ ]:
from scipy.signal import butter, filtfilt

FS = 360
BEFORE = int(0.4 * FS)
AFTER  = int(0.8 * FS)

def bandpass_filter(signal, fs=360):
    nyquist = fs / 2

    b, a = butter(
        3,
        [0.5 / nyquist, 40 / nyquist],
        btype="band"
    )

    return filtfilt(b, a, signal)


def extract_record(record_id):

    path = os.path.join(DATA_DIR, record_id)

    record = wfdb.rdrecord(path)
    annotation = wfdb.rdann(path, "atr")

    # Primary ECG channel
    signal = record.p_signal[:, 0]

    # Remove NaN if present
    signal = np.nan_to_num(signal)

    # ECG filtering
    signal = bandpass_filter(signal, fs=int(record.fs))

    X = []
    y = []
    record_ids = []

    for sample, symbol in zip(
        annotation.sample,
        annotation.symbol
    ):

        if symbol in NORMAL:
            label = 0

        elif symbol in ABNORMAL:
            label = 1

        else:
            continue

        start = sample - BEFORE
        end = sample + AFTER

        if start < 0 or end > len(signal):
            continue

        beat = signal[start:end]

        # Per-beat normalization
        beat = (
            beat - np.mean(beat)
        ) / (np.std(beat) + 1e-8)

        X.append(beat.astype(np.float32))
        y.append(label)
        record_ids.append(record_id)

    return (
        np.asarray(X),
        np.asarray(y),
        np.asarray(record_ids)
    )

In [ ]:
all_X = []
all_y = []
all_records = []

for i, record_id in enumerate(RECORDS):

    print(
        f"[{i+1:02d}/{len(RECORDS)}] "
        f"Processing record {record_id}..."
    )

    try:

        X_r, y_r, r_r = extract_record(record_id)

        all_X.append(X_r)
        all_y.append(y_r)
        all_records.append(r_r)

        print(
            "   Beats:",
            len(y_r),
            "| Normal:",
            np.sum(y_r == 0),
            "| Abnormal:",
            np.sum(y_r == 1)
        )

    except Exception as e:

        print(
            f"   ERROR in {record_id}:",
            str(e)
        )


X = np.concatenate(all_X)
y = np.concatenate(all_y)
record_ids = np.concatenate(all_records)

print("\n" + "="*50)
print("FINAL DATASET")
print("="*50)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(Counter(y))

print("\nNormal:", np.sum(y == 0))
print("Abnormal:", np.sum(y == 1))

print("\nNumber of records:",
      len(np.unique(record_ids)))

In [ ]:
import matplotlib.pyplot as plt

normal_idx = np.where(y == 0)[0][0]
abnormal_idx = np.where(y == 1)[0][0]

time = np.arange(X.shape[1]) / FS

plt.figure(figsize=(12,4))
plt.plot(time, X[normal_idx])
plt.xlabel("Time (s)")
plt.ylabel("Normalized amplitude")
plt.title("Example Normal ECG Beat")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
plt.plot(time, X[abnormal_idx])
plt.xlabel("Time (s)")
plt.ylabel("Normalized amplitude")
plt.title("Example Abnormal ECG Beat")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 70% train, 30% temporary
split1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(
        X,
        y,
        groups=record_ids
    )
)

X_train = X[train_idx]
y_train = y[train_idx]
r_train = record_ids[train_idx]

X_temp = X[temp_idx]
y_temp = y[temp_idx]
r_temp = record_ids[temp_idx]


# Split remaining 30% into 15% validation / 15% test
split2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=42
)

val_sub_idx, test_sub_idx = next(
    split2.split(
        X_temp,
        y_temp,
        groups=r_temp
    )
)

X_val = X_temp[val_sub_idx]
y_val = y_temp[val_sub_idx]
r_val = r_temp[val_sub_idx]

X_test = X_temp[test_sub_idx]
y_test = y_temp[test_sub_idx]
r_test = r_temp[test_sub_idx]


print("="*50)
print("RECORD-LEVEL SPLIT")
print("="*50)

print(
    "Train:",
    X_train.shape,
    Counter(y_train)
)

print(
    "Validation:",
    X_val.shape,
    Counter(y_val)
)

print(
    "Test:",
    X_test.shape,
    Counter(y_test)
)

print("\nTrain records:",
      sorted(np.unique(r_train)))

print("\nValidation records:",
      sorted(np.unique(r_val)))

print("\nTest records:",
      sorted(np.unique(r_test)))


# Verify leakage
assert set(r_train).isdisjoint(set(r_val))
assert set(r_train).isdisjoint(set(r_test))
assert set(r_val).isdisjoint(set(r_test))

print("\n✓ No record leakage detected.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class ECGDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(
            X,
            dtype=torch.float32
        ).unsqueeze(1)

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = ECGDataset(
    X_train,
    y_train
)

val_dataset = ECGDataset(
    X_val,
    y_val
)

test_dataset = ECGDataset(
    X_test,
    y_test
)

In [ ]:
class_counts = np.bincount(y_train)

class_weights = (
    1.0 / class_counts
)

sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=torch.tensor(
        sample_weights,
        dtype=torch.double
    ),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=2
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import torch.nn as nn

class ECGCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv1d(
                1, 32,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(
                32, 64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(
                64, 128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        x = self.features(x)

        return self.classifier(x)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = ECGCNN().to(DEVICE)

print(model)

parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("\nParameters:", parameters)
print("Device:", DEVICE)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


def train_one_epoch(model, loader):

    model.train()

    total_loss = 0

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(
            logits,
            yb
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * xb.size(0)
        )

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    ys = []
    preds = []
    probs = []

    for xb, yb in loader:

        xb = xb.to(DEVICE)

        logits = model(xb)

        probability = torch.softmax(
            logits,
            dim=1
        )[:,1]

        prediction = logits.argmax(dim=1)

        ys.extend(yb.numpy())

        preds.extend(
            prediction.cpu().numpy()
        )

        probs.extend(
            probability.cpu().numpy()
        )

    ys = np.asarray(ys)
    preds = np.asarray(preds)
    probs = np.asarray(probs)

    return {
        "accuracy":
            accuracy_score(ys, preds),

        "precision":
            precision_score(
                ys,
                preds,
                zero_division=0
            ),

        "recall":
            recall_score(
                ys,
                preds,
                zero_division=0
            ),

        "f1":
            f1_score(
                ys,
                preds,
                zero_division=0
            ),

        "auroc":
            roc_auc_score(
                ys,
                probs
            )
    }

In [ ]:
import copy

EPOCHS = 15
PATIENCE = 4

best_f1 = -1
best_state = None
patience_counter = 0

history = []

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Loss={train_loss:.4f} | "
        f"Val Acc={val_metrics['accuracy']:.4f} | "
        f"Val Recall={val_metrics['recall']:.4f} | "
        f"Val F1={val_metrics['f1']:.4f} | "
        f"Val AUROC={val_metrics['auroc']:.4f}"
    )

    if val_metrics["f1"] > best_f1:

        best_f1 = val_metrics["f1"]

        best_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("\nEarly stopping.")

            break


model.load_state_dict(best_state)

print("\nBest validation F1:", best_f1)

In [ ]:
test_metrics = evaluate(
    model,
    test_loader
)

print("="*50)
print("CLEAN TEST RESULTS")
print("="*50)

for metric, value in test_metrics.items():

    print(
        f"{metric.upper():10s}: "
        f"{value:.4f}"
    )

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

model.eval()

all_true = []
all_pred = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(DEVICE)

        logits = model(xb)

        pred = logits.argmax(
            dim=1
        ).cpu().numpy()

        all_true.extend(
            yb.numpy()
        )

        all_pred.extend(pred)


cm = confusion_matrix(
    all_true,
    all_pred
)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")

print(
    classification_report(
        all_true,
        all_pred,
        target_names=[
            "Normal",
            "Abnormal"
        ],
        digits=4
    )
)

In [ ]:
clean_results = {
    "accuracy": 0.8768,
    "precision": 0.3784,
    "recall": 0.7916,
    "f1": 0.5121,
    "auroc": 0.8575
}

torch.save(
    model.state_dict(),
    "/content/medstride_ecg_model.pth"
)

print("Model saved.")

In [ ]:
def fgsm_attack(model, x, y, epsilon):

    x = x.clone().detach().to(DEVICE)
    y = y.clone().detach().to(DEVICE)

    x.requires_grad = True

    logits = model(x)
    loss = criterion(logits, y)

    model.zero_grad()
    loss.backward()

    gradient = x.grad.detach()

    x_adv = x + epsilon * gradient.sign()

    return x_adv.detach()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_fgsm(model, loader, epsilon):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    abnormal_total = 0
    abnormal_clean_correct = 0
    successful_suppression = 0

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        # Clean predictions
        with torch.no_grad():
            clean_logits = model(xb)
            clean_pred = clean_logits.argmax(dim=1)

        # Generate adversarial samples
        xb_adv = fgsm_attack(
            model,
            xb,
            yb,
            epsilon
        )

        with torch.no_grad():

            adv_logits = model(xb_adv)

            adv_prob = torch.softmax(
                adv_logits,
                dim=1
            )[:, 1]

            adv_pred = adv_logits.argmax(dim=1)

        # Abnormal beats
        abnormal_mask = (yb == 1)

        abnormal_total += abnormal_mask.sum().item()

        # Beats correctly recognized as abnormal before attack
        clean_correct_mask = (
            abnormal_mask &
            (clean_pred == 1)
        )

        abnormal_clean_correct += (
            clean_correct_mask.sum().item()
        )

        # Clinically important attack:
        # abnormal was correctly detected before,
        # but becomes normal after attack
        suppression_mask = (
            clean_correct_mask &
            (adv_pred == 0)
        )

        successful_suppression += (
            suppression_mask.sum().item()
        )

        y_true.extend(
            yb.cpu().numpy()
        )

        y_pred.extend(
            adv_pred.cpu().numpy()
        )

        y_prob.extend(
            adv_prob.cpu().numpy()
        )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    ASR = (
        successful_suppression /
        abnormal_clean_correct
        if abnormal_clean_correct > 0
        else 0
    )

    return {

        "epsilon": epsilon,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "auroc":
            roc_auc_score(
                y_true,
                y_prob
            ),

        "clean_detected_abnormal":
            abnormal_clean_correct,

        "suppressed_alerts":
            successful_suppression,

        "attack_success_rate":
            ASR
    }

In [ ]:
EPSILONS = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.10
]

fgsm_results = []

for eps in EPSILONS:

    print(
        f"Running FGSM epsilon={eps}..."
    )

    result = evaluate_fgsm(
        model,
        test_loader,
        eps
    )

    fgsm_results.append(result)

fgsm_df = pd.DataFrame(
    fgsm_results
)

fgsm_df

In [ ]:
print(
    fgsm_df[
        [
            "epsilon",
            "accuracy",
            "recall",
            "f1",
            "auroc",
            "suppressed_alerts",
            "attack_success_rate"
        ]
    ].to_string(index=False)
)

In [ ]:
fgsm_df.to_csv(
    "/content/T4_2_FGSM_results.csv",
    index=False
)

In [ ]:
def pgd_attack(
    model,
    x,
    y,
    epsilon,
    alpha,
    steps=10
):

    original = (
        x.clone()
        .detach()
        .to(DEVICE)
    )

    y = y.to(DEVICE)

    # Random initialization
    adv = original.clone().detach()

    adv += torch.empty_like(
        adv
    ).uniform_(
        -epsilon,
        epsilon
    )

    for _ in range(steps):

        adv.requires_grad = True

        logits = model(adv)

        loss = criterion(
            logits,
            y
        )

        model.zero_grad()

        loss.backward()

        gradient = (
            adv.grad
            .detach()
        )

        adv = (
            adv.detach()
            + alpha *
            gradient.sign()
        )

        # Project back into epsilon ball
        delta = torch.clamp(
            adv - original,
            min=-epsilon,
            max=epsilon
        )

        adv = (
            original + delta
        ).detach()

    return adv

In [ ]:
def evaluate_pgd(
    model,
    loader,
    epsilon,
    alpha,
    steps=10
):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    abnormal_clean_correct = 0
    successful_suppression = 0

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.no_grad():

            clean_logits = model(xb)

            clean_pred = (
                clean_logits.argmax(
                    dim=1
                )
            )

        xb_adv = pgd_attack(
            model,
            xb,
            yb,
            epsilon,
            alpha,
            steps
        )

        with torch.no_grad():

            logits = model(xb_adv)

            probs = torch.softmax(
                logits,
                dim=1
            )[:,1]

            pred = logits.argmax(
                dim=1
            )

        clean_correct_abnormal = (
            (yb == 1) &
            (clean_pred == 1)
        )

        suppression = (
            clean_correct_abnormal &
            (pred == 0)
        )

        abnormal_clean_correct += (
            clean_correct_abnormal
            .sum()
            .item()
        )

        successful_suppression += (
            suppression
            .sum()
            .item()
        )

        y_true.extend(
            yb.cpu().numpy()
        )

        y_pred.extend(
            pred.cpu().numpy()
        )

        y_prob.extend(
            probs.cpu().numpy()
        )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    ASR = (
        successful_suppression /
        abnormal_clean_correct
    )

    return {

        "epsilon": epsilon,
        "alpha": alpha,
        "steps": steps,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "auroc":
            roc_auc_score(
                y_true,
                y_prob
            ),

        "clean_detected_abnormal":
            abnormal_clean_correct,

        "suppressed_alerts":
            successful_suppression,

        "attack_success_rate":
            ASR
    }

In [ ]:
PGD_EPSILONS = [
    0.005,
    0.01,
    0.02,
    0.05,
    0.10
]

pgd_results = []

for eps in PGD_EPSILONS:

    alpha = eps / 4

    print(
        f"Running PGD "
        f"epsilon={eps}, "
        f"alpha={alpha:.5f}"
    )

    result = evaluate_pgd(
        model,
        test_loader,
        epsilon=eps,
        alpha=alpha,
        steps=10
    )

    pgd_results.append(
        result
    )


pgd_df = pd.DataFrame(
    pgd_results
)

print(
    pgd_df[
        [
            "epsilon",
            "accuracy",
            "recall",
            "f1",
            "auroc",
            "suppressed_alerts",
            "attack_success_rate"
        ]
    ].to_string(index=False)
)

pgd_df.to_csv(
    "/content/T4_2_PGD_results.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    fgsm_df["epsilon"],
    fgsm_df["attack_success_rate"],
    marker="o",
    label="FGSM"
)

plt.plot(
    pgd_df["epsilon"],
    pgd_df["attack_success_rate"],
    marker="s",
    label="PGD"
)

plt.xlabel(
    "Perturbation Magnitude (ε)"
)

plt.ylabel(
    "Alert Suppression Attack Success Rate"
)

plt.title(
    "T4.2 — Adversarial Alert Suppression"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "/content/T4_2_attack_success.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
AGENTS = [
    "Diagnostic Agent",
    "Treatment Agent",
    "Medication Agent",
    "Execution Layer"
]

TOTAL_AGENTS = len(AGENTS)

In [ ]:
ARCHITECTURES = [
    "Fully Autonomous",
    "Safety Validator",
    "Human-in-the-Loop"
]

In [ ]:
def clinical_cascade(
    true_label,
    diagnostic_prediction,
    architecture,
    safety_detection_probability=0.90,
    human_detection_probability=0.98,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng(42)

    # We care specifically about abnormal patient
    # incorrectly classified as normal.

    dangerous_miss = (
        true_label == 1
        and diagnostic_prediction == 0
    )

    if not dangerous_miss:

        return {
            "cascade": 0,
            "cascade_depth": 0,
            "affected_agents": 0,
            "unsafe_action": 0,
            "intercepted": 0
        }

    # Initial diagnostic compromise
    affected = 1

    # --------------------------------
    # Fully autonomous architecture
    # --------------------------------

    if architecture == "Fully Autonomous":

        # Treatment agent trusts diagnosis
        affected += 1

        # Medication agent acts
        affected += 1

        # Execution layer receives action
        affected += 1

        return {
            "cascade": 1,
            "cascade_depth": 4,
            "affected_agents": affected,
            "unsafe_action": 1,
            "intercepted": 0
        }

    # --------------------------------
    # Safety-validator architecture
    # --------------------------------

    elif architecture == "Safety Validator":

        detected = (
            rng.random()
            < safety_detection_probability
        )

        if detected:

            return {
                "cascade": 1,
                "cascade_depth": 2,
                "affected_agents": 2,
                "unsafe_action": 0,
                "intercepted": 1
            }

        else:

            return {
                "cascade": 1,
                "cascade_depth": 4,
                "affected_agents": 4,
                "unsafe_action": 1,
                "intercepted": 0
            }

    # --------------------------------
    # Human-in-the-loop
    # --------------------------------

    elif architecture == "Human-in-the-Loop":

        detected = (
            rng.random()
            < human_detection_probability
        )

        if detected:

            return {
                "cascade": 1,
                "cascade_depth": 2,
                "affected_agents": 2,
                "unsafe_action": 0,
                "intercepted": 1
            }

        else:

            return {
                "cascade": 1,
                "cascade_depth": 4,
                "affected_agents": 4,
                "unsafe_action": 1,
                "intercepted": 0
            }

    else:

        raise ValueError(
            "Unknown architecture"
        )

In [ ]:
SELECTED_EPSILON = 0.05

In [ ]:
def get_pgd_predictions(
    model,
    loader,
    epsilon=0.05
):

    model.eval()

    true_labels = []
    clean_predictions = []
    adv_predictions = []

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.no_grad():

            clean_logits = model(xb)

            clean_pred = (
                clean_logits
                .argmax(dim=1)
            )

        adv_x = pgd_attack(
            model,
            xb,
            yb,
            epsilon=epsilon,
            alpha=epsilon/4,
            steps=10
        )

        with torch.no_grad():

            adv_logits = model(
                adv_x
            )

            adv_pred = (
                adv_logits
                .argmax(dim=1)
            )

        true_labels.extend(
            yb.cpu().numpy()
        )

        clean_predictions.extend(
            clean_pred.cpu().numpy()
        )

        adv_predictions.extend(
            adv_pred.cpu().numpy()
        )

    return (
        np.asarray(true_labels),
        np.asarray(clean_predictions),
        np.asarray(adv_predictions)
    )


y_true_adv, clean_pred_adv, pgd_pred = (
    get_pgd_predictions(
        model,
        test_loader,
        epsilon=SELECTED_EPSILON
    )
)

print(
    "Samples:",
    len(y_true_adv)
)

In [ ]:
attack_induced_mask = (
    (y_true_adv == 1) &
    (clean_pred_adv == 1) &
    (pgd_pred == 0)
)

attack_indices = np.where(
    attack_induced_mask
)[0]

print(
    "Attack-induced alert suppressions:",
    len(attack_indices)
)

In [ ]:
cascade_results = []

rng = np.random.default_rng(42)

for architecture in ARCHITECTURES:

    for idx in attack_indices:

        result = clinical_cascade(
            true_label=y_true_adv[idx],
            diagnostic_prediction=pgd_pred[idx],
            architecture=architecture,
            rng=rng
        )

        result["architecture"] = (
            architecture
        )

        cascade_results.append(
            result
        )


cascade_df = pd.DataFrame(
    cascade_results
)

cascade_df.head()

In [ ]:
summary_rows = []

for architecture in ARCHITECTURES:

    subset = cascade_df[
        cascade_df["architecture"]
        == architecture
    ]

    trials = len(subset)

    cascades = subset[
        "cascade"
    ].sum()

    unsafe = subset[
        "unsafe_action"
    ].sum()

    intercepted = subset[
        "intercepted"
    ].sum()

    mean_depth = subset[
        "cascade_depth"
    ].mean()

    propagation_reach = (
        subset["affected_agents"]
        .mean()
        / TOTAL_AGENTS
    )

    UAR = (
        unsafe / trials
        if trials
        else 0
    )

    SIR = (
        intercepted / trials
        if trials
        else 0
    )

    summary_rows.append({

        "Architecture":
            architecture,

        "Attack Trials":
            trials,

        "Unsafe Actions":
            unsafe,

        "Unsafe Action Rate":
            UAR,

        "Mean Cascade Depth":
            mean_depth,

        "Propagation Reach":
            propagation_reach,

        "Safety Interception Rate":
            SIR
    })


cascade_summary = pd.DataFrame(
    summary_rows
)

print(
    cascade_summary.to_string(
        index=False
    )
)

cascade_summary.to_csv(
    "/content/T4_2_cascade_summary.csv",
    index=False
)

In [ ]:
DETECTION_PROBS = np.arange(
    0.0,
    1.01,
    0.1
)

In [ ]:
sensitivity_results = []

N_REPEATS = 100

for detection_prob in DETECTION_PROBS:

    unsafe_rates = []

    for repeat in range(N_REPEATS):

        rng = np.random.default_rng(
            1000 + repeat
        )

        unsafe = 0

        for idx in attack_indices:

            result = clinical_cascade(
                true_label=y_true_adv[idx],
                diagnostic_prediction=pgd_pred[idx],
                architecture="Safety Validator",
                safety_detection_probability=detection_prob,
                rng=rng
            )

            unsafe += result[
                "unsafe_action"
            ]

        unsafe_rate = (
            unsafe /
            len(attack_indices)
            if len(attack_indices)
            else 0
        )

        unsafe_rates.append(
            unsafe_rate
        )

    sensitivity_results.append({

        "Detection Probability":
            detection_prob,

        "Mean Unsafe Action Rate":
            np.mean(unsafe_rates),

        "SD":
            np.std(unsafe_rates)
    })


sensitivity_df = pd.DataFrame(
    sensitivity_results
)

print(
    sensitivity_df.to_string(
        index=False
    )
)

In [ ]:
plt.figure(figsize=(8,5))

plt.errorbar(
    sensitivity_df[
        "Detection Probability"
    ],
    sensitivity_df[
        "Mean Unsafe Action Rate"
    ],
    yerr=sensitivity_df["SD"],
    marker="o",
    capsize=3
)

plt.xlabel(
    "Safety Validator Detection Probability"
)

plt.ylabel(
    "Unsafe Action Rate"
)

plt.title(
    "T4.2 Cascade Mitigation Sensitivity Analysis"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "/content/T4_2_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def targeted_pgd_attack(
    model,
    x,
    epsilon,
    alpha,
    steps=10
):
    """
    Targeted PGD:
    forces ECG samples toward class 0 (Normal).
    """

    original = x.clone().detach().to(DEVICE)

    # Target class = NORMAL
    target = torch.zeros(
        original.size(0),
        dtype=torch.long,
        device=DEVICE
    )

    # deterministic initialization:
    # start exactly from original input
    adv = original.clone().detach()

    for _ in range(steps):

        adv.requires_grad_(True)

        logits = model(adv)

        loss = criterion(
            logits,
            target
        )

        model.zero_grad()
        loss.backward()

        gradient = adv.grad.detach()

        # TARGETED attack:
        # minimize loss toward target class
        adv = (
            adv.detach()
            - alpha * gradient.sign()
        )

        delta = torch.clamp(
            adv - original,
            min=-epsilon,
            max=epsilon
        )

        adv = (
            original + delta
        ).detach()

    return adv

In [ ]:
def evaluate_targeted_pgd(
    model,
    loader,
    epsilon,
    steps=10
):

    model.eval()

    total_clean_detected = 0
    successfully_suppressed = 0

    perturbation_linf = []
    perturbation_l2 = []

    probabilities_before = []
    probabilities_after = []

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.no_grad():

            clean_logits = model(xb)

            clean_probs = torch.softmax(
                clean_logits,
                dim=1
            )[:, 1]

            clean_pred = clean_logits.argmax(
                dim=1
            )

        # Only abnormal samples that were
        # correctly detected before attack
        mask = (
            (yb == 1) &
            (clean_pred == 1)
        )

        if mask.sum() == 0:
            continue

        x_selected = xb[mask]

        before_probs = clean_probs[mask]

        x_adv = targeted_pgd_attack(
            model,
            x_selected,
            epsilon=epsilon,
            alpha=epsilon / 4,
            steps=steps
        )

        with torch.no_grad():

            adv_logits = model(x_adv)

            adv_probs = torch.softmax(
                adv_logits,
                dim=1
            )[:, 1]

            adv_pred = adv_logits.argmax(
                dim=1
            )

        # Successful suppression:
        # abnormal -> normal
        successful = (
            adv_pred == 0
        )

        total_clean_detected += len(
            x_selected
        )

        successfully_suppressed += (
            successful.sum().item()
        )

        delta = (
            x_adv - x_selected
        ).detach()

        linf = (
            delta.abs()
            .view(delta.size(0), -1)
            .max(dim=1)[0]
        )

        l2 = (
            delta
            .view(delta.size(0), -1)
            .norm(p=2, dim=1)
        )

        perturbation_linf.extend(
            linf.cpu().numpy()
        )

        perturbation_l2.extend(
            l2.cpu().numpy()
        )

        probabilities_before.extend(
            before_probs.cpu().numpy()
        )

        probabilities_after.extend(
            adv_probs.cpu().numpy()
        )

    asr = (
        successfully_suppressed /
        total_clean_detected
    )

    return {

        "epsilon": epsilon,

        "clean_detected_abnormal":
            total_clean_detected,

        "suppressed_alerts":
            successfully_suppressed,

        "targeted_ASR":
            asr,

        "mean_Linf":
            np.mean(perturbation_linf),

        "mean_L2":
            np.mean(perturbation_l2),

        "mean_abnormal_prob_before":
            np.mean(probabilities_before),

        "mean_abnormal_prob_after":
            np.mean(probabilities_after)
    }

In [ ]:
TARGET_EPS = [
    0.001,
    0.0025,
    0.005,
    0.0075,
    0.010,
    0.015,
    0.020,
    0.025,
    0.030
]

targeted_results = []

for eps in TARGET_EPS:

    print(
        f"Targeted PGD ε={eps:.4f}"
    )

    result = evaluate_targeted_pgd(
        model,
        test_loader,
        epsilon=eps,
        steps=10
    )

    targeted_results.append(result)


targeted_df = pd.DataFrame(
    targeted_results
)

print(
    targeted_df.to_string(
        index=False
    )
)

targeted_df.to_csv(
    "/content/T4_2_targeted_PGD.csv",
    index=False
)

In [ ]:
def get_attack_example(
    model,
    loader,
    epsilon=0.01
):

    model.eval()

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.no_grad():

            clean_logits = model(xb)

            clean_pred = clean_logits.argmax(
                dim=1
            )

        mask = (
            (yb == 1) &
            (clean_pred == 1)
        )

        if mask.sum() == 0:
            continue

        selected = xb[mask]

        adv = targeted_pgd_attack(
            model,
            selected,
            epsilon=epsilon,
            alpha=epsilon/4,
            steps=10
        )

        with torch.no_grad():

            adv_pred = model(
                adv
            ).argmax(dim=1)

        success = (
            adv_pred == 0
        )

        if success.sum() > 0:

            idx = torch.where(
                success
            )[0][0]

            return (
                selected[idx]
                .squeeze()
                .cpu()
                .numpy(),

                adv[idx]
                .squeeze()
                .cpu()
                .numpy()
            )

    return None, None

In [ ]:
clean_ecg, adv_ecg = get_attack_example(
    model,
    test_loader,
    epsilon=0.01
)

time = np.arange(
    len(clean_ecg)
) / FS

plt.figure(figsize=(10, 4))

plt.plot(
    time,
    clean_ecg,
    label="Clean ECG"
)

plt.plot(
    time,
    adv_ecg,
    label="Adversarial ECG",
    alpha=0.75
)

plt.xlabel("Time (s)")
plt.ylabel("Normalized amplitude")

plt.title(
    "Clean vs Targeted Adversarial ECG"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "/content/ECG_clean_vs_adversarial.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
perturbation = (
    adv_ecg - clean_ecg
)

plt.figure(figsize=(10, 3))

plt.plot(
    time,
    perturbation
)

plt.xlabel("Time (s)")
plt.ylabel("Perturbation")

plt.title(
    "Adversarial Perturbation"
)

plt.tight_layout()

plt.savefig(
    "/content/ECG_perturbation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Maximum absolute perturbation:",
    np.max(
        np.abs(perturbation)
    )
)

print(
    "RMSE:",
    np.sqrt(
        np.mean(
            perturbation ** 2
        )
    )
)

In [ ]:
import torch
import torch.nn as nn

class ECGSafetyBiLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # CNN input:
        # [batch, 1, time]
        #
        # LSTM expects:
        # [batch, time, features]

        x = x.transpose(1, 2)

        output, _ = self.lstm(x)

        # Global mean pooling over time
        x = output.mean(dim=1)

        return self.classifier(x)


safety_model = ECGSafetyBiLSTM().to(DEVICE)

print(safety_model)

print(
    "\nSafety model parameters:",
    sum(
        p.numel()
        for p in safety_model.parameters()
    )
)

print("Device:", DEVICE)

In [ ]:
safety_criterion = nn.CrossEntropyLoss()

safety_optimizer = torch.optim.AdamW(
    safety_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)


def train_safety_epoch(model, loader):

    model.train()

    running_loss = 0

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        safety_optimizer.zero_grad()

        logits = model(xb)

        loss = safety_criterion(
            logits,
            yb
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        safety_optimizer.step()

        running_loss += (
            loss.item() *
            xb.size(0)
        )

    return (
        running_loss /
        len(loader.dataset)
    )

In [ ]:
import copy

SAFETY_EPOCHS = 15
PATIENCE = 4

best_safety_f1 = -1
best_safety_state = None
counter = 0

for epoch in range(
    1,
    SAFETY_EPOCHS + 1
):

    loss = train_safety_epoch(
        safety_model,
        train_loader
    )

    metrics = evaluate(
        safety_model,
        val_loader
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss={loss:.4f} | "
        f"Acc={metrics['accuracy']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"AUROC={metrics['auroc']:.4f}"
    )

    if metrics["f1"] > best_safety_f1:

        best_safety_f1 = metrics["f1"]

        best_safety_state = copy.deepcopy(
            safety_model.state_dict()
        )

        counter = 0

    else:

        counter += 1

        if counter >= PATIENCE:
            print("Early stopping.")
            break


safety_model.load_state_dict(
    best_safety_state
)

torch.save(
    safety_model.state_dict(),
    "/content/medstride_safety_bilstm.pth"
)

print(
    "\nBest validation F1:",
    best_safety_f1
)

In [ ]:
safety_test_metrics = evaluate(
    safety_model,
    test_loader
)

print("=" * 55)
print("SAFETY AGENT — CLEAN TEST PERFORMANCE")
print("=" * 55)

for name, value in safety_test_metrics.items():

    print(
        f"{name.upper():10s}: "
        f"{value:.4f}"
    )

In [ ]:
FINAL_EPSILONS = [
    0.005,
    0.010,
    0.015,
    0.020
]

In [ ]:
def evaluate_safety_transfer(
    primary_model,
    safety_model,
    loader,
    epsilon
):

    primary_model.eval()
    safety_model.eval()

    eligible = 0
    primary_suppressed = 0
    safety_detected = 0
    unsafe_after_safety = 0

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        # --------------------------------
        # Primary model — clean prediction
        # --------------------------------

        with torch.no_grad():

            primary_clean_logits = (
                primary_model(xb)
            )

            primary_clean_pred = (
                primary_clean_logits
                .argmax(dim=1)
            )

        # Eligible:
        # truly abnormal AND
        # primary originally detects it
        eligible_mask = (
            (yb == 1) &
            (primary_clean_pred == 1)
        )

        if eligible_mask.sum() == 0:
            continue

        x_selected = xb[
            eligible_mask
        ]

        eligible += (
            x_selected.size(0)
        )

        # --------------------------------
        # Attack primary diagnostic agent
        # --------------------------------

        x_adv = targeted_pgd_attack(
            primary_model,
            x_selected,
            epsilon=epsilon,
            alpha=epsilon / 4,
            steps=10
        )

        with torch.no_grad():

            primary_adv_pred = (
                primary_model(x_adv)
                .argmax(dim=1)
            )

        # Attack successfully suppresses
        # primary alert
        suppressed_mask = (
            primary_adv_pred == 0
        )

        n_suppressed = (
            suppressed_mask
            .sum()
            .item()
        )

        primary_suppressed += (
            n_suppressed
        )

        if n_suppressed == 0:
            continue

        compromised_ecgs = (
            x_adv[suppressed_mask]
        )

        # --------------------------------
        # Independent safety agent
        # --------------------------------

        with torch.no_grad():

            safety_logits = (
                safety_model(
                    compromised_ecgs
                )
            )

            safety_pred = (
                safety_logits
                .argmax(dim=1)
            )

        # Safety model says abnormal
        detected = (
            safety_pred == 1
        )

        n_detected = (
            detected.sum().item()
        )

        safety_detected += (
            n_detected
        )

        unsafe_after_safety += (
            n_suppressed -
            n_detected
        )

    ASR = (
        primary_suppressed / eligible
        if eligible
        else 0
    )

    SIR = (
        safety_detected /
        primary_suppressed
        if primary_suppressed
        else 0
    )

    residual_UAR = (
        unsafe_after_safety /
        primary_suppressed
        if primary_suppressed
        else 0
    )

    return {

        "epsilon":
            epsilon,

        "eligible_abnormal":
            eligible,

        "primary_suppressed":
            primary_suppressed,

        "primary_ASR":
            ASR,

        "safety_intercepted":
            safety_detected,

        "safety_interception_rate":
            SIR,

        "residual_unsafe":
            unsafe_after_safety,

        "residual_unsafe_rate":
            residual_UAR
    }

In [ ]:
transfer_results = []

for eps in FINAL_EPSILONS:

    print(
        f"Testing transfer at ε={eps}..."
    )

    result = evaluate_safety_transfer(
        model,
        safety_model,
        test_loader,
        epsilon=eps
    )

    transfer_results.append(
        result
    )


transfer_df = pd.DataFrame(
    transfer_results
)

print(
    transfer_df.to_string(
        index=False
    )
)

transfer_df.to_csv(
    "/content/T4_2_safety_transfer.csv",
    index=False
)

In [ ]:
def experimental_cascade_metrics(
    primary_model,
    safety_model,
    loader,
    epsilon
):

    primary_model.eval()
    safety_model.eval()

    rows = []

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.no_grad():

            clean_pred = (
                primary_model(xb)
                .argmax(dim=1)
            )

        eligible = (
            (yb == 1) &
            (clean_pred == 1)
        )

        if eligible.sum() == 0:
            continue

        selected = xb[eligible]

        adv = targeted_pgd_attack(
            primary_model,
            selected,
            epsilon=epsilon,
            alpha=epsilon/4,
            steps=10
        )

        with torch.no_grad():

            attacked_primary = (
                primary_model(adv)
                .argmax(dim=1)
            )

            safety_prediction = (
                safety_model(adv)
                .argmax(dim=1)
            )

        for primary_p, safety_p in zip(
            attacked_primary.cpu().numpy(),
            safety_prediction.cpu().numpy()
        ):

            # Attack failed
            if primary_p == 1:

                rows.append({
                    "attack_success": 0,
                    "cascade_depth_no_safety": 0,
                    "cascade_depth_safety": 0,
                    "unsafe_no_safety": 0,
                    "unsafe_with_safety": 0
                })

                continue

            # Attack succeeded:
            # primary says NORMAL

            # Without safety:
            # diagnostic -> treatment ->
            # medication -> execution
            depth_no_safety = 4
            unsafe_no_safety = 1

            # Safety agent independently
            # detects abnormality
            if safety_p == 1:

                depth_safety = 2
                unsafe_safety = 0

            else:

                depth_safety = 4
                unsafe_safety = 1

            rows.append({

                "attack_success": 1,

                "cascade_depth_no_safety":
                    depth_no_safety,

                "cascade_depth_safety":
                    depth_safety,

                "unsafe_no_safety":
                    unsafe_no_safety,

                "unsafe_with_safety":
                    unsafe_safety
            })

    return pd.DataFrame(rows)

In [ ]:
cascade_experimental_results = []

for eps in [
    0.010,
    0.015,
    0.020
]:

    df = experimental_cascade_metrics(
        model,
        safety_model,
        test_loader,
        epsilon=eps
    )

    attacked = df[
        df["attack_success"] == 1
    ]

    result = {

        "epsilon":
            eps,

        "successful_attacks":
            len(attacked),

        "UAR_without_safety":
            attacked[
                "unsafe_no_safety"
            ].mean(),

        "UAR_with_safety":
            attacked[
                "unsafe_with_safety"
            ].mean(),

        "mean_depth_without_safety":
            attacked[
                "cascade_depth_no_safety"
            ].mean(),

        "mean_depth_with_safety":
            attacked[
                "cascade_depth_safety"
            ].mean()
    }

    cascade_experimental_results.append(
        result
    )


experimental_summary = pd.DataFrame(
    cascade_experimental_results
)

print(
    experimental_summary.to_string(
        index=False
    )
)

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

def ecg_quality_metrics(clean, adv):

    clean = clean.flatten()
    adv = adv.flatten()

    noise = adv - clean

    rmse = np.sqrt(
        np.mean(noise ** 2)
    )

    signal_power = np.sum(
        clean ** 2
    )

    noise_power = np.sum(
        noise ** 2
    )

    snr = 10 * np.log10(
        signal_power /
        (noise_power + 1e-12)
    )

    prd = 100 * np.sqrt(
        noise_power /
        (signal_power + 1e-12)
    )

    correlation = pearsonr(
        clean,
        adv
    )[0]

    cosine = cosine_similarity(
        clean.reshape(1, -1),
        adv.reshape(1, -1)
    )[0, 0]

    return {
        "RMSE": rmse,
        "SNR_dB": snr,
        "PRD_percent": prd,
        "Correlation": correlation,
        "Cosine_similarity": cosine
    }

In [ ]:
def evaluate_ecg_quality(
    model,
    loader,
    epsilon
):

    model.eval()

    rows = []

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        with torch.no_grad():

            clean_pred = (
                model(xb)
                .argmax(dim=1)
            )

        eligible = (
            (yb == 1) &
            (clean_pred == 1)
        )

        if eligible.sum() == 0:
            continue

        clean = xb[eligible]

        adv = targeted_pgd_attack(
            model,
            clean,
            epsilon=epsilon,
            alpha=epsilon / 4,
            steps=10
        )

        with torch.no_grad():

            adv_pred = (
                model(adv)
                .argmax(dim=1)
            )

        success = (
            adv_pred == 0
        )

        clean_success = (
            clean[success]
            .detach()
            .cpu()
            .numpy()
        )

        adv_success = (
            adv[success]
            .detach()
            .cpu()
            .numpy()
        )

        for c, a in zip(
            clean_success,
            adv_success
        ):

            metrics = ecg_quality_metrics(
                c,
                a
            )

            rows.append(metrics)

    df = pd.DataFrame(rows)

    return df

In [ ]:
quality_results = []

for eps in [
    0.010,
    0.015,
    0.020
]:

    print(
        f"Evaluating ECG quality ε={eps}"
    )

    qdf = evaluate_ecg_quality(
        model,
        test_loader,
        eps
    )

    quality_results.append({

        "epsilon": eps,

        "successful_attacks":
            len(qdf),

        "RMSE_mean":
            qdf["RMSE"].mean(),

        "SNR_mean_dB":
            qdf["SNR_dB"].mean(),

        "PRD_mean_percent":
            qdf["PRD_percent"].mean(),

        "Correlation_mean":
            qdf["Correlation"].mean(),

        "Cosine_mean":
            qdf["Cosine_similarity"].mean()
    })


quality_summary = pd.DataFrame(
    quality_results
)

print(
    quality_summary.to_string(
        index=False
    )
)

quality_summary.to_csv(
    "/content/T4_2_ECG_quality.csv",
    index=False
)

In [ ]:
def bootstrap_proportion_ci(
    successes,
    total,
    n_bootstrap=10000,
    seed=42
):

    rng = np.random.default_rng(seed)

    observations = np.concatenate([
        np.ones(successes),
        np.zeros(total - successes)
    ])

    bootstrap_values = []

    for _ in range(n_bootstrap):

        sample = rng.choice(
            observations,
            size=total,
            replace=True
        )

        bootstrap_values.append(
            sample.mean()
        )

    lower = np.percentile(
        bootstrap_values,
        2.5
    )

    upper = np.percentile(
        bootstrap_values,
        97.5
    )

    return lower, upper

In [ ]:
ci_rows = []

for _, row in transfer_df.iterrows():

    asr_low, asr_high = (
        bootstrap_proportion_ci(
            int(row["primary_suppressed"]),
            int(row["eligible_abnormal"])
        )
    )

    sir_low, sir_high = (
        bootstrap_proportion_ci(
            int(row["safety_intercepted"]),
            int(row["primary_suppressed"])
        )
    )

    unsafe_low, unsafe_high = (
        bootstrap_proportion_ci(
            int(row["residual_unsafe"]),
            int(row["primary_suppressed"])
        )
    )

    ci_rows.append({

        "epsilon":
            row["epsilon"],

        "ASR":
            row["primary_ASR"],

        "ASR_CI_low":
            asr_low,

        "ASR_CI_high":
            asr_high,

        "SIR":
            row[
                "safety_interception_rate"
            ],

        "SIR_CI_low":
            sir_low,

        "SIR_CI_high":
            sir_high,

        "Residual_UAR":
            row[
                "residual_unsafe_rate"
            ],

        "UAR_CI_low":
            unsafe_low,

        "UAR_CI_high":
            unsafe_high
    })


ci_df = pd.DataFrame(
    ci_rows
)

print(
    ci_df.to_string(
        index=False
    )
)

ci_df.to_csv(
    "/content/T4_2_confidence_intervals.csv",
    index=False
)

In [ ]:
publication_table = transfer_df[
    [
        "epsilon",
        "primary_suppressed",
        "primary_ASR",
        "safety_intercepted",
        "safety_interception_rate",
        "residual_unsafe",
        "residual_unsafe_rate"
    ]
].copy()

publication_table[
    "primary_ASR"
] *= 100

publication_table[
    "safety_interception_rate"
] *= 100

publication_table[
    "residual_unsafe_rate"
] *= 100


publication_table.columns = [
    "ε",
    "Suppressed Alerts",
    "Attack Success (%)",
    "Safety Interceptions",
    "Interception (%)",
    "Residual Unsafe",
    "Residual UAR (%)"
]

print(
    publication_table.to_string(
        index=False
    )
)